# 02 - Baseline: hand-crafted features + XGBoost (Colab)

Features come from `vtac.features` (scipy: rates, RR variability, amplitude, high-frequency ratio,
cross-channel rate agreement, presence flags) **plus** neurokit2 HRV/QRS features computed here on the
same preprocessed windows the CNN sees. XGBoost handles NaN natively (missing channels).

Ablation rows (see `docs/modeling.md`): **B0** ECG features only, **B1** ECG + pulsatile features.
Thresholds are tuned on **validation only** (hard TPR >= 0.95, plus the unconstrained optimum); the
test set is evaluated **once**, in the last section.

> Status: written but **not yet executed on Colab**. The neurokit2 block in particular needs a
> first-run check (see the failure counter it prints).

In [ ]:
# ---- Setup: mount Drive, install vtac from GitHub (pinned to a ref) -------------------------
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

VTAC_REF = "main"  # replace with a tag/commit once stable (reproducibility!)
!pip -q install "vtac[train] @ git+https://github.com/siddhu-exe/vtac-alarm-monitor.git@{VTAC_REF}"

import vtac
from vtac.provenance import installed_git_ref
GIT_REF = installed_git_ref(VTAC_REF)
print("vtac", vtac.__version__, "ref", GIT_REF)

DRIVE = Path("/content/drive/MyDrive/vtac")
for sub in ("processed", "replay", "models", "reports"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import json, time
import numpy as np, pandas as pd
from vtac.config import VtacConfig
from vtac.dataset import load_windows
from vtac import metrics as M

OFFSET = 2.0  # 2.0 = project default (W1); rerun the notebook with 0.0 for the benchmark window (W0)
z = load_windows(DRIVE / "processed" / f"windows_off{OFFSET}.npz")
cfg = VtacConfig.from_dict(json.loads(str(z["config_json"])))
S, Mk, y, split, patient = z["signals"], z["mask"], z["y"].astype(int), z["split"], z["patient_id"]
idx = {s: np.flatnonzero(split == s) for s in ("train", "val", "test")}
print({k: (len(v), round(float(y[v].mean()), 3)) for k, v in idx.items()}, "| window", S.shape, "offset", cfg.window_end_offset_s)

## 1. Features

In [ ]:
from tqdm.auto import tqdm
from vtac.features import extract_features, feature_names

t0 = time.time()
X = pd.DataFrame([extract_features(S[i], Mk[i], cfg) for i in tqdm(range(len(S)))], columns=feature_names(cfg))
print("vtac features:", X.shape, f"{time.time()-t0:.0f}s")

In [ ]:
# neurokit2 HRV / R-peak features per ECG slot (computed only where the channel is present).
import neurokit2 as nk
NK_COLS = ["HRV_MeanNN", "HRV_SDNN", "HRV_RMSSD", "HRV_MedianNN", "HRV_CVNN", "HRV_MadNN"]
nk_errors = 0

def nk_ecg_features(sig, fs):
    global nk_errors
    out = dict.fromkeys(["nk_n_rpeaks"] + NK_COLS, np.nan)
    try:
        _, info = nk.ecg_peaks(sig, sampling_rate=fs)
        r = info["ECG_R_Peaks"]
        out["nk_n_rpeaks"] = float(len(r))
        if len(r) >= 4:
            hrv = nk.hrv_time(info, sampling_rate=fs)
            for c in NK_COLS:
                if c in hrv:
                    out[c] = float(hrv[c].iloc[0])
    except Exception:
        nk_errors += 1
    return out

t0 = time.time()
rows = []
for i in tqdm(range(len(S))):
    row = {}
    for c, slot in enumerate(cfg.channel_slots[:2]):  # ECG_1, ECG_2
        f = nk_ecg_features(S[i, c], cfg.fs) if Mk[i, c] > 0 else dict.fromkeys(["nk_n_rpeaks"] + NK_COLS, np.nan)
        row.update({f"{slot}_{k}": v for k, v in f.items()})
    rows.append(row)
X = pd.concat([X, pd.DataFrame(rows)], axis=1)
print(f"neurokit2 features done in {time.time()-t0:.0f}s | exceptions swallowed: {nk_errors} | NaN rate:",
      X.filter(like="HRV_").isna().mean().round(2).to_dict())
X.to_pickle(DRIVE / "processed" / f"baseline_features_off{OFFSET}.pkl")

## 2. Train B0 / B1 (validation only)

In [ ]:
import xgboost as xgb

FEATURE_SETS = {
    "B0_ecg_feats": [c for c in X.columns if c.startswith("ECG_") or c == "rate_diff_ecg1_ecg2"],
    "B1_all_feats": list(X.columns),
}
tr, va, te = idx["train"], idx["val"], idx["test"]
spw = float((y[tr] == 0).sum() / (y[tr] == 1).sum())
print("scale_pos_weight", round(spw, 2))

fitted, val_rows = {}, []
for name, cols in FEATURE_SETS.items():
    clf = xgb.XGBClassifier(
        n_estimators=1500, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=2, scale_pos_weight=spw, eval_metric="aucpr", early_stopping_rounds=60,
        tree_method="hist", random_state=SEED, n_jobs=-1)
    clf.fit(X.loc[tr, cols], y[tr], eval_set=[(X.loc[va, cols], y[va])], verbose=False)
    p_va = clf.predict_proba(X.loc[va, cols])[:, 1]
    tuned = M.tune_thresholds(y[va], p_va, tpr_target=cfg.tpr_target)
    fitted[name] = dict(clf=clf, cols=cols, tuned=tuned)
    for kind in ("constrained", "unconstrained"):
        r = tuned[kind]
        val_rows.append(dict(run=name, point=kind, thr=round(r["threshold"], 4), pr_auc=r["pr_auc"], roc_auc=r["roc_auc"],
                             tpr=r["tpr"], ppv=r["ppv"], tnr=r["tnr"], challenge=r["challenge_score"]))
    print(name, "best_iter", clf.best_iteration)
val_table = pd.DataFrame(val_rows).round(4)
val_table

In [ ]:
imp = pd.Series(fitted["B1_all_feats"]["clf"].feature_importances_, index=FEATURE_SETS["B1_all_feats"]).sort_values(ascending=False)
print(imp.head(15).round(4))

## 3. FINAL - test evaluation (run once)

Uses the thresholds chosen on validation. Reports both operating points, patient-level bootstrap
95% CIs, and the split by channel availability (records with a pulsatile channel vs ECG-only).

In [ ]:
has_puls = (Mk[:, 2] > 0) | (Mk[:, 3] > 0)
final = {}
for name, f in fitted.items():
    p_te = f["clf"].predict_proba(X.loc[te, f["cols"]])[:, 1]
    res = {}
    for kind in ("constrained", "unconstrained"):
        thr = f["tuned"][kind]["threshold"]
        ev = M.evaluate(y[te], p_te, thr)
        lo, hi = M.group_bootstrap_ci(lambda a, b, t=thr: M.evaluate(a, b, t)["challenge_score"], y[te], p_te, patient[te], n_boot=500, seed=SEED)
        ev["challenge_ci95"] = (lo, hi)
        res[kind] = ev
    res["pr_auc_ci95"] = M.group_bootstrap_ci(M.pr_auc, y[te], p_te, patient[te], n_boot=500, seed=SEED)
    sub = {}
    for label, m in (("with_pulsatile", has_puls[te]), ("ecg_only", ~has_puls[te])):
        if m.sum() > 10 and 0 < y[te][m].sum() < m.sum():
            sub[label] = dict(n=int(m.sum()), pr_auc=M.pr_auc(y[te][m], p_te[m]), roc_auc=M.roc_auc(y[te][m], p_te[m]))
    res["by_channels"] = sub
    final[name] = res
    print(f"== {name} (test)  PR-AUC {res['constrained']['pr_auc']:.3f} {tuple(round(v,3) for v in res['pr_auc_ci95'])}  ROC-AUC {res['constrained']['roc_auc']:.3f}")
    for kind in ("constrained", "unconstrained"):
        r = res[kind]
        print(f"   {kind:13s} thr {r['threshold']:.3f}  TPR {r['tpr']:.3f}  PPV {r['ppv']:.3f}  TNR {r['tnr']:.3f}  "
              f"score {r['challenge_score']:.3f} {tuple(round(v,3) for v in r['challenge_ci95'])}  FN {r['fn']} FP {r['fp']}")
    print("   by channels:", sub)

out = DRIVE / "reports" / f"baseline_metrics_off{OFFSET}.json"
out.write_text(json.dumps(dict(val=val_rows, test=final, offset=OFFSET, vtac_ref=GIT_REF, seed=SEED), indent=2, default=float))
print("saved", out)

In [ ]:
# Persist the B1 model for reference (the served model is the CNN from 03_cnn).
fitted["B1_all_feats"]["clf"].save_model(str(DRIVE / "models" / f"xgb_baseline_off{OFFSET}.ubj"))